In [ ]:
"""
=============================================================
FILE 38 — TOOLFORMER PATTERN
=============================================================

CONCEPTS TAUGHT
----------------
1. Toolformer Pattern
2. Autonomous Tool Selection
3. Tool-Aware Reasoning
4. Dynamic Tool Usage
5. AI Tool Learning
6. Tool Calling Systems
7. Multi-Step Tool Usage
8. Agentic Tool Planning
9. Reasoning + Action Systems
10. Tool-Augmented Intelligence

CORE IDEA
-----------
The AI autonomously decides:
- whether tools are needed
- which tools to use
- when to use them

FLOW
-----
Question
   ↓
Reason About Tools
   ↓
Select Tool
   ↓
Execute Tool
   ↓
Continue Reasoning

REAL WORLD USE CASES
---------------------
- AI copilots
- coding agents
- research assistants
- autonomous AI systems
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict
from typing import Annotated

from langgraph.graph.message import add_messages

from langgraph.graph import StateGraph, START

from langgraph.prebuilt import ToolNode, tools_condition

from IPython.display import Image, display

# ============================================================
# STEP 2 — ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — DEFINE TOOLS
# ============================================================

def calculator(a: int, b: int) -> int:
    """
    Add two numbers.
    """
    return a + b

def currency_converter(
    amount: float
) -> float:
    """
    Convert USD to INR.
    """
    return amount * 83

tools = [
    calculator,
    currency_converter
]

# ============================================================
# STEP 5 — BIND TOOLS
# ============================================================

llm_with_tools = llm.bind_tools(tools)

# ============================================================
# STEP 6 — STATE
# ============================================================

class State(TypedDict):
    messages: Annotated[list, add_messages]

# ============================================================
# STEP 7 — TOOLFORMER AGENT
# ============================================================

def toolformer_agent(state: State):

    """
    AI autonomously reasons
    about tool usage.
    """

    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }

# ============================================================
# STEP 8 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node(
    "toolformer_agent",
    toolformer_agent
)

builder.add_node(
    "tools",
    ToolNode(tools)
)

# ============================================================
# STEP 9 — DEFINE EDGES
# ============================================================

builder.add_edge(
    START,
    "toolformer_agent"
)

builder.add_conditional_edges(
    "toolformer_agent",
    tools_condition
)

builder.add_edge(
    "tools",
    "toolformer_agent"
)

# ============================================================
# STEP 10 — COMPILE
# ============================================================

graph = builder.compile()

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 11 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "messages":
        [
            (
                "human",
                """
                Add 100 and 250.
                Then convert 350 USD into INR.
                """
            )
        ]
    }
)

# ============================================================
# STEP 12 — PRINT RESULT
# ============================================================

print("\nFINAL AGENT RESPONSE\n")
print("=" * 60)

for msg in result["messages"]:
    print(msg)